In [ ]:
import asyncio
import json
import os
import dotenv
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    executor,
    tool,
)
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential
from IPython.display import HTML, display
from pydantic import BaseModel

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

print("All imports successful!")

## Paso 1: Definir modelos Pydantic para salidas estructuradas

Estos modelos definen el **esquema** que los agentes devolverán. Usar `response_format` con Pydantic garantiza:
- ✅ Extracción de datos con seguridad de tipo
- ✅ Validación automática
- ✅ Sin errores de análisis por respuestas en texto libre
- ✅ Enrutamiento condicional sencillo basado en campos


In [ ]:
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""

    destination: str
    has_availability: bool
    message: str


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""

    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""

    destination: str
    action: str
    message: str


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")

## Paso 2: Crear la Herramienta de Reserva de Hotel

Esta herramienta es la que el **availability_agent** llamará para comprobar si hay habitaciones disponibles. Usamos el decorador `@ai_function` para:
- Convertir una función de Python en una herramienta que pueda ser llamada por IA
- Generar automáticamente el esquema JSON para el LLM
- Manejar la validación de parámetros
- Habilitar la invocación automática por agentes

Para esta demostración:
- **Estocolmo, Seattle, Tokio, Londres, Ámsterdam** → Tienen habitaciones ✅
- **Todas las demás ciudades** → No tienen habitaciones ❌


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.

    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

## Paso 3: Definir funciones de condición para el enrutamiento

Estas funciones inspeccionan la respuesta del agente y determinan qué camino tomar en el flujo de trabajo.

**Patrón clave:**
1. Comprobar si el mensaje es `AgentExecutorResponse`
2. Analizar la salida estructurada (modelo Pydantic)
3. Devolver `True` o `False` para controlar el enrutamiento

El flujo de trabajo evaluará estas condiciones en los **bordes** para decidir qué ejecutor invocar a continuación.


In [ ]:
def has_availability_condition(message: Any) -> bool:
    """
    Condition for routing when hotels ARE available.
    
    Returns True if the destination has hotel rooms.
    """
    if not isinstance(message, AgentExecutorResponse):
        return True  # Default to True if unexpected type

    try:
        result = BookingCheckResult.model_validate_json(message.agent_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}
            </div>
        """)
        )

        return result.has_availability
    except Exception as e:
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                <strong>⚠️  Error:</strong> {str(e)}
            </div>
        """)
        )
        return False


def no_availability_condition(message: Any) -> bool:
    """
    Condition for routing when hotels are NOT available.
    
    Returns True if the destination has no hotel rooms.
    """
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )

        return not result.has_availability
    except Exception as e:
        return False


print("✅ Condition functions defined:")
print("   - has_availability_condition (routes when rooms exist)")
print("   - no_availability_condition (routes when no rooms)")

## Paso 4: Crear un Executor de Visualización Personalizado

Los executors son componentes del flujo de trabajo que realizan transformaciones o efectos secundarios. Usamos el decorador `@executor` para crear un executor personalizado que muestra el resultado final.

**Conceptos clave:**
- `@executor(id="...")` - Registra una función como un executor de flujo de trabajo
- `WorkflowContext[Never, str]` - Indicaciones de tipo para entrada/salida
- `ctx.yield_output(...)` - Produce el resultado final del flujo de trabajo


In [ ]:
@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """
    Display the final result as workflow output.
    
    This executor receives the final agent response and yields it as the workflow output.
    """
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_response.text)


print("✅ display_result executor created with @executor decorator")

## Paso 5: Cargar Variables de Entorno

Configure el cliente LLM. Este ejemplo funciona con:
- **Modelos de GitHub** (nivel gratuito con token de GitHub)
- **Azure OpenAI**
- **OpenAI**


In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Paso 6: Crear agentes de IA con salidas estructuradas

Creamos **tres agentes especializados**, cada uno envuelto en un `AgentExecutor`:

1. **availability_agent** - Verifica la disponibilidad del hotel usando la herramienta
2. **alternative_agent** - Sugiere ciudades alternativas (cuando no hay habitaciones)
3. **booking_agent** - Anima a reservar (cuando hay habitaciones disponibles)

**Características clave:**
- `tools=[hotel_booking]` - Proporciona la herramienta al agente
- `response_format=PydanticModel` - Obliga a una salida JSON estructurada
- `AgentExecutor(..., id="...")` - Envuelve al agente para uso en flujos de trabajo


In [ ]:
# Agent 1: Check availability with tool
availability_agent = AgentExecutor(
    client.as_agent(
        name="availability-agent",
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), and message (string). "
            "The message should summarize the availability status."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
    ),
    id="availability_agent",
)

alternative_agent = AgentExecutor(
    client.as_agent(
        name="alternative-agent",
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

booking_agent = AgentExecutor(
    client.as_agent(
        name="booking-agent",
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created 3 Agents:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - Checks availability with hotel_booking tool</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
        </ul>
    </div>
""")
)

## Paso 7: Construir el Flujo de Trabajo con Bordes Condicionales

Ahora usamos `WorkflowBuilder` para construir el grafo con enrutamiento condicional:

**Estructura del flujo de trabajo:**
```
availability_agent (START)
        ↓
   Evaluate conditions
        ↙         ↘
[no_availability]  [has_availability]
        ↓              ↓
alternative_agent  booking_agent
        ↓              ↓
    display_result ←───┘
```

**Métodos clave:**
- `.set_start_executor(...)` - Establece el punto de entrada
- `.add_edge(from, to, condition=...)` - Agrega borde condicional
- `.build()` - Finaliza el flujo de trabajo


In [ ]:
# Build the workflow with conditional routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    # NO AVAILABILITY PATH
    .add_edge(availability_agent, alternative_agent, condition=no_availability_condition)
    .add_edge(alternative_agent, display_result)
    # HAS AVAILABILITY PATH
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Conditional Routing:</strong><br>
            • If <strong>NO availability</strong> → alternative_agent → display_result<br>
            • If <strong>availability</strong> → booking_agent → display_result
        </p>
    </div>
""")
)

## Paso 8: Ejecutar Caso de Prueba 1 - Ciudad SIN Disponibilidad (París)

Probemos la ruta de **sin disponibilidad** solicitando hoteles en París (que no tiene habitaciones en nuestra simulación).


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Paris (No Availability)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → alternative_agent → display_result</p>
    </div>
""")
)

# Create request for Paris
request_paris = AgentExecutorRequest(
    messages=[Message(role="user", contents="I want to book a hotel in Paris")]
)

# Run the workflow
events_paris = await workflow.run(request_paris)
outputs_paris = events_paris.get_outputs()

# Display results
if outputs_paris:
    result_paris = AlternativeResult.model_validate_json(outputs_paris[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px; box-shadow: 0 4px 12px rgba(255,165,0,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 WORKFLOW RESULT (Paris)</h3>
            <div style='background: white; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Alternative Suggestion:</strong> 🏨 {result_paris.alternative_destination}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Reason:</strong> {result_paris.reason}</p>
            </div>
        </div>
    """)
    )

## Paso 9: Ejecutar Caso de Prueba 2 - Ciudad CON Disponibilidad (Estocolmo)

Ahora vamos a probar la ruta de **disponibilidad** solicitando hoteles en Estocolmo (que tiene habitaciones en nuestra simulación).


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 2: Stockholm (Has Availability)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → booking_agent → display_result</p>
    </div>
""")
)

# Create request for Stockholm
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", contents="I want to book a hotel in Stockholm")]
)

# Run the workflow
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT (Stockholm)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
            </div>
        </div>
    """)
    )

## Puntos Clave y Próximos Pasos

### ✅ Lo que Has Aprendido:

1. **Patrón WorkflowBuilder**
   - Usa `.set_start_executor()` para definir el punto de entrada
   - Usa `.add_edge(from, to, condition=...)` para enrutamiento condicional
   - Llama a `.build()` para finalizar el flujo de trabajo

2. **Enrutamiento Condicional**
   - Las funciones de condición inspeccionan `AgentExecutorResponse`
   - Analizan salidas estructuradas para tomar decisiones de enrutamiento
   - Devuelven `True` para activar una arista, `False` para omitirla

3. **Integración de Herramientas**
   - Usa `@ai_function` para convertir funciones de Python en herramientas AI
   - Los agentes llaman a las herramientas automáticamente cuando es necesario
   - Las herramientas devuelven JSON que los agentes pueden analizar

4. **Salidas Estructuradas**
   - Usa modelos Pydantic para extracción de datos con tipos seguros
   - Establece `response_format=MyModel` al crear agentes
   - Analiza respuestas con `Model.model_validate_json()`

5. **Ejecutores Personalizados**
   - Usa `@executor(id="...")` para crear componentes del flujo de trabajo
   - Los ejecutores pueden transformar datos o realizar efectos secundarios
   - Usa `ctx.yield_output()` para producir resultados en el flujo de trabajo

### 🚀 Aplicaciones en el Mundo Real:

- **Reservas de Viaje**: Verificar disponibilidad, sugerir alternativas, comparar opciones
- **Atención al Cliente**: Enrutamiento según tipo de problema, sentimiento, prioridad
- **Comercio Electrónico**: Verificar inventario, sugerir alternativas, procesar pedidos
- **Moderación de Contenido**: Enrutamiento según puntuaciones de toxicidad, reportes de usuarios
- **Flujos de Aprobación**: Enrutamiento según monto, rol de usuario, nivel de riesgo
- **Procesamiento en Múltiples Etapas**: Enrutamiento según calidad y completitud de datos

### 📚 Próximos Pasos:

- Añadir condiciones más complejas (múltiples criterios)
- Implementar bucles con gestión del estado del flujo de trabajo
- Añadir sub-flujos para componentes reutilizables
- Integrar con APIs reales (reservas de hotel, sistemas de inventario)
- Añadir manejo de errores y rutas alternativas
- Visualizar flujos con las herramientas de visualización integradas


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Descargo de responsabilidad**:
Este documento ha sido traducido utilizando el servicio de traducción automática [Co-op Translator](https://github.com/Azure/co-op-translator). Aunque nos esforzamos por la precisión, tenga en cuenta que las traducciones automatizadas pueden contener errores o inexactitudes. El documento original en su idioma nativo debe considerarse la fuente autorizada. Para información crítica, se recomienda una traducción profesional humana. No somos responsables de cualquier malentendido o interpretación errónea que surja del uso de esta traducción.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
